# 08 — MNIST Digit Recognition

We apply everything we've built so far to a real problem:  
teach the network to read handwritten digits (0–9).

**Dataset:** MNIST — 70,000 grayscale images, 28×28 pixels each  
**Task:** classify each image into one of 10 classes  

**What's new:**
- `DataLoader` — feeds data in batches during training
- `Softmax` — turns raw scores (logits) into probabilities across 10 classes
- `CrossEntropyLoss` — the standard loss for multi-class classification
- Visualisations: sample images, training curves, predictions, confusion matrix

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Load the data

In [ ]:
# Transforms: convert PNG → Tensor, then normalise pixel values
# 0.1307 and 0.3081 are the MNIST dataset mean/std (pre-computed)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST(root='data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST(root='data', train=False, download=True, transform=transform)

# DataLoader: shuffles and batches the data automatically
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=256, shuffle=False)

print(f'Training samples: {len(train_data):,}')
print(f'Test samples:     {len(test_data):,}')
print(f'Each image: {train_data[0][0].shape}  (channels × height × width)')

## Preview: what does the data look like?

In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(15, 5))
for cls in range(10):
    # Pick 3 examples of each digit
    indices = [i for i, (_, label) in enumerate(train_data) if label == cls][:3]
    for row, idx in enumerate(indices):
        img, label = train_data[idx]
        axes[row, cls].imshow(img.squeeze(), cmap='gray')
        axes[row, cls].axis('off')
        if row == 0:
            axes[row, cls].set_title(str(cls), fontsize=12)

fig.suptitle('Sample training images — 3 examples of each digit', fontsize=13)
plt.tight_layout()
plt.savefig('08_samples.png', dpi=120)
plt.show()

## What is the network actually seeing? Flattening an image

In [ ]:
img, label = train_data[0]
print(f'Label: {label}')
print(f'Image shape: {img.shape}  → 28×28 = 784 pixels')

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].imshow(img.squeeze(), cmap='gray')
axes[0].set_title(f'Image (28×28)  label={label}')
axes[0].axis('off')

axes[1].plot(img.numpy().ravel(), color='steelblue', linewidth=0.5)
axes[1].set_title('Same image flattened to 784 numbers — what the network sees')
axes[1].set_xlabel('Pixel index (0–783)')
axes[1].set_ylabel('Pixel value (normalised)')

plt.tight_layout()
plt.show()

## Define the network

We use a fully-connected network with two hidden layers.  
The final layer has 10 outputs — one score (logit) per digit class.

```
784 pixels → 256 neurons → 128 neurons → 10 scores
```

**ReLU** instead of sigmoid in the hidden layers — it trains faster  
**CrossEntropyLoss** = softmax + log + negative mean (standard for classification)

In [ ]:
class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),          # 1×28×28 → 784
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),    # 10 raw scores (logits)
        )

    def forward(self, x):
        return self.net(x)


model = MNISTNet().to(device)
print(model)
total = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total:,}')

## Train

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def evaluate(loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
    return correct / len(loader.dataset)


EPOCHS = 5
train_losses, train_accs, test_accs = [], [], []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)   # same 3-step loop as before
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_acc = evaluate(train_loader)
    test_acc  = evaluate(test_loader)

    train_losses.append(avg_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)

    print(f'Epoch {epoch+1}/{EPOCHS}  loss={avg_loss:.4f}  '
          f'train={train_acc:.1%}  test={test_acc:.1%}')

torch.save(model.state_dict(), 'mnist_model.pt')
print('\nModel saved to mnist_model.pt')

## Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, EPOCHS+1), train_losses, marker='o', color='darkorange')
ax1.set_title('Training loss per epoch')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('CrossEntropy loss')

ax2.plot(range(1, EPOCHS+1), train_accs, marker='o', label='Train', color='steelblue')
ax2.plot(range(1, EPOCHS+1), test_accs,  marker='s', label='Test',  color='crimson')
ax2.set_title('Accuracy per epoch')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax2.legend()

plt.tight_layout()
plt.savefig('08_training.png', dpi=120)
plt.show()

## Sample predictions — see what the network actually thinks

In [ ]:
model.eval()
imgs_show, labels_show = next(iter(test_loader))
imgs_show = imgs_show[:20]
labels_show = labels_show[:20]

with torch.no_grad():
    logits = model(imgs_show.to(device)).cpu()
    probs  = torch.softmax(logits, dim=1)
    preds  = probs.argmax(dim=1)
    confidences = probs.max(dim=1).values

fig, axes = plt.subplots(4, 5, figsize=(13, 11))
for i, ax in enumerate(axes.ravel()):
    img   = imgs_show[i].squeeze().numpy()
    pred  = preds[i].item()
    truth = labels_show[i].item()
    conf  = confidences[i].item()
    correct = pred == truth

    ax.imshow(img, cmap='gray')
    ax.axis('off')
    color = 'green' if correct else 'red'
    ax.set_title(f'pred={pred}  ({conf:.0%})\ntrue={truth}',
                 color=color, fontsize=9)

fig.suptitle('Green = correct, Red = wrong', fontsize=12)
plt.tight_layout()
plt.savefig('08_predictions.png', dpi=120)
plt.show()

## Output probabilities for a single image

The network gives a probability for **each** of the 10 classes.  
A confident network assigns nearly 100% to one digit and ~0% to the rest.

In [ ]:
idx = 0   # change this to inspect different test images
img_single, true_label = test_data[idx]

with torch.no_grad():
    logit = model(img_single.unsqueeze(0).to(device)).cpu()
    prob  = torch.softmax(logit, dim=1).squeeze().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.imshow(img_single.squeeze(), cmap='gray')
ax1.set_title(f'Input image   (true label: {true_label})')
ax1.axis('off')

bar_colors = ['crimson' if i == prob.argmax() else 'steelblue' for i in range(10)]
ax2.bar(range(10), prob, color=bar_colors)
ax2.set_xticks(range(10))
ax2.set_xlabel('Digit class')
ax2.set_ylabel('Probability')
ax2.set_title('Network output probabilities\n(red bar = predicted class)')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))

plt.tight_layout()
plt.savefig('08_single_prob.png', dpi=120)
plt.show()
print(f'True label: {true_label}   Predicted: {prob.argmax()}   Confidence: {prob.max():.1%}')

## Confusion Matrix — which digits get confused with which?

In [ ]:
all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        preds_batch = model(imgs.to(device)).argmax(dim=1).cpu()
        all_preds.extend(preds_batch.numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=range(10), yticklabels=range(10))
ax.set_xlabel('Predicted digit')
ax.set_ylabel('True digit')
ax.set_title('Confusion matrix on 10,000 test images\n'
             '(diagonal = correct, off-diagonal = mistakes)')
plt.tight_layout()
plt.savefig('08_confusion.png', dpi=120)
plt.show()

# What does the network confuse most?
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
worst = np.unravel_index(cm_no_diag.argmax(), cm_no_diag.shape)
print(f'Most common mistake: true={worst[0]} predicted as {worst[1]}  ({cm_no_diag[worst]} times)')

## Key Takeaways

- 5 epochs, ~1 min of training → >97% accuracy on 10,000 unseen images
- The 3-step loop (`zero_grad → loss → backward → step`) is unchanged from XOR
- `softmax` converts raw scores to probabilities summing to 1
- The confusion matrix shows which mistakes the model makes (3 ↔ 5, 4 ↔ 9 are common)

---

Next: interact with the model live — draw digits in a browser and watch the probabilities change in real time.  
→ `09_digit_explorer.py`